# BJ-235 — Causal Beijing handicap + pre-event SP time series recovery

Contract: `BJ-UNIFIED-BJ-SP-TIMESERIES-001 / v1` (PI-user design freeze 2026-08-16).

Goal: provenance-valid time series of Beijing native handicap lines and pre-event `3/1/0` SP quotes on the canonical identity/schema frozen by #234 (source-schema predecessor).

Boundaries: no probability model, no EU translation, no ticket construction, no ROI, no outcome opening.

## 0. Environment and binding

This notebook is executed from the delivery worktree; it reads the canonical SQLite store built by `scripts/bj235_build.py` and the recoverability report it produced. Data root points at the external data store (gitignored).

In [ ]:
import json, sys, sqlite3
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

CANON = ROOT / "data" / "derived" / "bj235" / "bj_sp_canonical.sqlite"
OUT = ROOT / "results" / "bj235_bj_sp_timeseries_v1"
report_path = OUT / "RECOVERABILITY_REPORT.json"

con = sqlite3.connect(f"file:{CANON}?mode=ro", uri=True)
con.row_factory = sqlite3.Row
report = json.loads(report_path.read_text(encoding="utf-8"))
print("canonical db:", CANON)
print("report loaded:", report_path)

## 1. Source registry (REQ-BJ01: every accepted quote binds source + clock)

Registered in the canonical store with admissible-use class and clock semantics per #234.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)
src = pd.read_sql_query("SELECT source_id, provider, clock_semantics, admissible_use, first_usable_at_utc FROM source_registry", con)
src

## 2. Raw presence waterfall (REQ-BJ06)

From raw source rows through identity join, line state, timestamp gate, complete triplet and final causal intersection.

In [ ]:
wf = report["waterfall_raw"]
ct = report["complete_triplets"]
print("RAW PRESENCE (source rows):")
for k, v in wf.items():
    print(f"  {k:32s} {v:>12,}")
print()
print("COMPLETE 3/1/0 TRIPLETS (strict causal observations):")
for k, v in ct.items():
    print(f"  {k:32s} {v:>12,}")
print()
print("AS-OF REPLAY STATS:")
for k, v in report["build_steps"].get("asof", {}).items():
    print(f"  {k:32s} {v:>12,}")

## 3. Issue-level support by cohort (REQ-BJ06/REQ-BJ08)

Strict historical (2014–2025), retrospective proxy, and prospective (2026-05+) reported separately.

In [ ]:
support = report["issue_support"]
print("ISSUE-LEVEL SUPPORT BY COHORT:")
for cohort, s in support["by_cohort"].items():
    print(f"  {cohort:20s} issues={s['issues']:4d}  fixtures_total={s['fixtures_total']:6d}  fixtures_complete_asof={s['fixtures_complete_asof']:6d}")
print()
per_issue = pd.DataFrame(support["per_issue"])
per_issue.tail(15)

## 4. Historical recoverability evidence (REQ-BJ08)

Every historical (pre-2026) source is checked for a native, verifiable pre-event SP observation clock. None exists → strict historical cohort disposition.

In [ ]:
hist = report["historical"]
print(f"DISPOSITION: {hist['disposition']}")
print(f"strict_historical_causal_support: {hist['strict_historical_causal_support']}")
print(f"prospective source: {hist['prospective_source']}")
print(f"prospective clock: {hist['prospective_clock']}")
print()
for s in hist["strict_historical_sources_checked"]:
    print(f"  - {s['source_id']:28s} [{s['admissible_use']}] {s['why_not_causal']}")

## 5. Line state retention (REQ-BJ04)

Handicap lines are preserved per observation; as-of selection never joins across different native-line states.

In [ ]:
line_q = """
SELECT mo.handicap_line, COUNT(*) n_obs, COUNT(DISTINCT mo.fixture_id) n_fixtures
FROM market_observation mo
WHERE mo.source_id='bjdc_500_xml_34'
GROUP BY mo.handicap_line ORDER BY n_obs DESC
"""
lines = pd.read_sql_query(line_q, con)
lines

# line states per fixture: how many fixtures ever had >1 distinct line observed
multi = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT fixture_id FROM market_observation
        WHERE source_id='bjdc_500_xml_34' AND handicap_line IS NOT NULL
        GROUP BY fixture_id HAVING COUNT(DISTINCT handicap_line) > 1
    )
""").fetchone()[0]
print("fixtures with >1 observed native-line state:", multi)

## 6. Probability separation (REQ-BJ05)

Raw SP, normalized inverse-SP probability and winning/final SP are physically separate tables. No `0.65`, no settlement SP in inference tables.

In [ ]:
pe = report["build_steps"].get("probabilities", {})
print("probability rows built:", pe.get("probability_rows", 0))
bad_065 = con.execute("SELECT COUNT(*) FROM market_quote WHERE decimal_odds=0.65").fetchone()[0]
bad_settle = con.execute(
    "SELECT COUNT(*) FROM probability_estimate pe JOIN settlement s ON s.fixture_id=pe.fixture_id"
).fetchone()[0]
print("quotes == 0.65:", bad_065)
print("probability rows joined to settlement:", bad_settle)

prob = pd.read_sql_query("""
    SELECT fixture_id, outcome_code, ROUND(probability,4) p, observed_at_utc
    FROM probability_estimate ORDER BY fixture_id LIMIT 9
""", con)
prob

## 7. Exclusion ledger (REQ-BJ02: incomplete observations remain explicit)

Non-causal rows are not silently dropped — they are registered with reason codes.

In [ ]:
excl = pd.read_sql_query("""
    SELECT reason_code, COUNT(*) n, SUM(raw_count) total_raw
    FROM exclusion_ledger GROUP BY reason_code ORDER BY n DESC
""", con)
excl

## 8. Package integrity

Immutable Parquet snapshot + MANIFEST.json + SHA256SUMS.json under `results/bj235_bj_sp_timeseries_v1/`.

In [ ]:
manifest = json.loads((OUT / "MANIFEST.json").read_text(encoding="utf-8"))
print("package:", manifest["package"])
for name, t in manifest["tables"].items():
    print(f"  {name:24s} {t['rows']:>12,} rows  {t['file']}")
sums = json.loads((OUT / "SHA256SUMS.json").read_text(encoding="utf-8"))
print("hash-bound files:", len(sums))

## 9. Acceptance trace

| REQ | Evidence |
|---|---|
| BJ01 source+causal clock | source_registry; observations carry observed_at/received_at + source_id |
| BJ02 complete/finite/positive | as_of complete flag; incomplete→exclusion ledger; probability check p>0 |
| BJ03 no post-decision/kickoff | replay gates; post_decision/post_kickoff rejected counters; golden tests |
| BJ04 line changes retained | per-observation handicap_line; line-state test |
| BJ05 0.65/settlement absent | physical table separation; checks above |
| BJ06 cohort support | issue-level by cohort in §3 |
| BJ07 idempotent ingestion | INSERT OR IGNORE + batch keys; dup test |
| BJ08 terminal disposition | UNRECOVERABLE_HISTORICAL_BJ_SP evidence + prospective counts |

Disposition: **prospective cohort recovered (STRICT_CAUSAL); strict historical pre-event SP = `UNRECOVERABLE_HISTORICAL_BJ_SP`** (evidence-backed).